# dice (Dice)

In [ ]:
///- --test static

:< testing

In [ ]:
:< sm'_operators
:< digest

## face

In [ ]:
:| face =
    | One
    | Two
    | Three
    | Four
    | Five
    | Six

:= digit \* t {number}. (f : face) : t =
    conv (!!!!UnionTag(f) : i32)

:= pips (f : face) : u8 =
    digit f + 1u8

:= face_of_digit (d : i32) : face =
    ?= d ?|
    | 0 => One
    | 1 => Two
    | 2 => Three
    | 3 => Four
    | 4 => Five
    | _ => Six

:= face_of_pips (p : u8) : option face =
    ? p >= 1u8 && p <= 6u8 ?> Some (face_of_digit (conv (p - 1u8))) |: None

:= face_of_byte (b : u8) : option face =
    ? b < 252u8 ?> Some (face_of_digit (conv (b % 6u8))) |: None

#### every_face_round_trips_through_its_pips (test)

In [1]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

:= ~low = 0u8
(
    listm.map (pips >> face_of_pips) [ One; Two; Three; Four; Five; Six ]
    |> listm.map (optionm.map pips),
    face_of_pips low,
    face_of_pips (low + 7u8)
)
|> _assert_eq ([ Some 1u8; Some 2; Some 3; Some 4; Some 5; Some 6 ], None, None)

.rs (rust)
__assert_eq { expected = (Cons(Some(1), Cons(Some(2), Cons(Some(3), Cons(Some(4), Cons(Some(5), Cons(Some(6), Nil)))))), None, None) }
.fsx (fsharp)
__assert_eq { expected = (Cons(Some(1), Cons(Some(2), Cons(Some(3), Cons(Some(4), Cons(Some(5), Cons(Some(6), Nil)))))), None, None) }
.ts (typescript)
__assert_eq { expected = (Cons(Some(1), Cons(Some(2), Cons(Some(3), Cons(Some(4), Cons(Some(5), Cons(Some(6), Nil)))))), None, None) }
.py (python)
__assert_eq { expected = (Cons(Some(1), Cons(Some(2), Cons(Some(3), Cons(Some(4), Cons(Some(5), Cons(Some(6), Nil)))))), None, None) }
.lua (lua)
__assert_eq { expected = (Cons(Some(1), Cons(Some(2), Cons(Some(3), Cons(Some(4), Cons(Some(5), Cons(Some(6), Nil)))))), None, None) }
.c (c)
__assert_eq { expected = (Cons(Some(1), Cons(Some(2), Cons(Some(3), Cons(Some(4), Cons(Some(5), Cons(Some(6), Nil)))))), None, None) }
.zig (zig)
__assert_eq { expected = (Cons(Some(1), Cons(Some(2), Cons(Some(3), Cons(Some(4), Cons(Some(5), Cons(Some(

0

In [ ]:
:= face_counts () : array i32 =
    := counts : array i32 = create 7
    :~ ^ clear (i : i32) : () =
        ? i < 7 ?>
            set counts i 0
            clear (i + 1)
    clear 0
    :~ ^ go (b : i32) : () =
        ? b <= 255 ?>
            := slot =
                ?= face_of_byte (conv b) ?|
                | Some f => digit f
                | None => 6i32
            set counts slot (index counts slot + 1)
            go (b + 1)
    go 0
    counts

#### every_face_takes_the_same_share_of_the_bytes (test)

In [2]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

face_counts ()
|> _assert_eq (;[ 42i32; 42; 42; 42; 42; 42; 4 ] : array i32)

.rs (rust)
__assert_eq { expected = [42, 42, 42, 42, 42, 42, 4] }
.fsx (fsharp)
__assert_eq { expected = [42, 42, 42, 42, 42, 42, 4] }
.ts (typescript)
__assert_eq { expected = [42, 42, 42, 42, 42, 42, 4] }
.py (python)
__assert_eq { expected = [42, 42, 42, 42, 42, 42, 4] }
.lua (lua)
__assert_eq { expected = [42, 42, 42, 42, 42, 42, 4] }
.c (c)
__assert_eq { expected = [42, 42, 42, 42, 42, 42, 4] }
.zig (zig)
__assert_eq { expected = [42, 42, 42, 42, 42, 42, 4] }
.gleam (gleam)
__assert_eq { expected = [42, 42, 42, 42, 42, 42, 4] }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

## bound

In [ ]:
:! bound t = t

:| bound_error t =
    | EmptyRange
    | AboveLargest : t * t

:= ^ largest_from \* t {number}. (p : t) : t =
    ? p > (limit.max : t) / 6 ?> p |: largest_from (p * 6)

:= largest_bound \* t {number}. () : t =
    largest_from 1

:= bound_of \* t {number}. (max : t) : result (bound t) (bound_error t) =
    ? max < 1 ?> Error EmptyRange
    |? max > largest_bound () ?> Error (AboveLargest (max, largest_bound ()))
    |: Ok (bound max)

:= bound_value \* t. ((bound max) : bound t) : t =
    max

:= bound_error_text \* t {number}. (e : bound_error t) : string =
    ?= e ?|
    | EmptyRange => "max must be at least 1"
    | AboveLargest (max, largest) => "max: " ++# #?max ++# " is above the largest supported bound " ++# #?largest

:= dice_count \* t {number}. (b : bound t) : i32 =
    := max = bound_value b
    :~ ^ go (count : i32) (reach : t) : i32 =
        ? reach >= max ?> count |: go (count + 1) (reach * 6)
    ? max = 1 ?> 1 |: go 0 1

#### largest_bound_is_the_largest_power_of_six_of_the_type (test)

In [3]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

((largest_bound () : i32), (largest_bound () : u64), (largest_bound () : u8))
|> _assert_eq (362797056i32, 4738381338321616896u64, 216u8)

.rs (rust)
__assert_eq { expected = (362797056, 4738381338321616896, 216) }
.fsx (fsharp)
__assert_eq { expected = (362797056, 4738381338321616896, 216) }
.ts (typescript)
__assert_eq { expected = (362797056, 4738381338321616896, 216) }
.py (python)
__assert_eq { expected = (362797056, 4738381338321616896, 216) }
.lua (lua)
__assert_eq { expected = (362797056, 4738381338321616896, 216) }
.c (c)
__assert_eq { expected = (362797056, 4738381338321616896, 216) }
.zig (zig)
__assert_eq { expected = (362797056, 4738381338321616896, 216) }
.gleam (gleam)
__assert_eq { expected = (362797056, 4738381338321616896, 216) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### bound_of_rejects_empty_and_oversized_ranges (test)

In [4]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

:= ~zero = 0u64
:= ~over = 4738381338321616897u64
:= text (r : result (bound u64) (bound_error u64)) =
    ?= r ?|
    | Ok b => #?(bound_value b)
    | Error e => bound_error_text e
(text (bound_of zero), text (bound_of over), text (bound_of (zero + 2000u64)))
|> _assert_eq (
    "max must be at least 1",
    "max: 4738381338321616897 is above the largest supported bound 4738381338321616896",
    "2000"
)

.rs (rust)
__assert_eq { expected = (max must be at least 1, max: 4738381338321616897 is above the largest supported bound 4738381338321616896, 2000) }
.fsx (fsharp)
__assert_eq { expected = (max must be at least 1, max: 4738381338321616897 is above the largest supported bound 4738381338321616896, 2000) }
.ts (typescript)
__assert_eq { expected = (max must be at least 1, max: 4738381338321616897 is above the largest supported bound 4738381338321616896, 2000) }
.py (python)
__assert_eq { expected = (max must be at least 1, max: 4738381338321616897 is above the largest supported bound 4738381338321616896, 2000) }
.lua (lua)
__assert_eq { expected = (max must be at least 1, max: 4738381338321616897 is above the largest supported bound 4738381338321616896, 2000) }
.c (c)
__assert_eq { expected = (max must be at least 1, max: 4738381338321616897 is above the largest supported bound 4738381338321616896, 2000) }
.zig (zig)
__assert_eq { expected = (max must be at least 1, max: 473838133832161

0

In [ ]:
:= count_of \* t {number}. (max : t) : i32 =
    ?= bound_of max ?|
    | Ok b => dice_count b
    | Error _ => -1

#### dice_count_is_the_fewest_dice_that_reach_the_bound (test)

In [5]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

:= ~one = 1i32
(count_of one, count_of (one * 36), count_of (one * 7777), count_of (one * 362797056), count_of 4738381338321616896u64)
|> _assert_eq (1i32, 2i32, 6i32, 11i32, 24i32)

.rs (rust)
__assert_eq { expected = (1, 2, 6, 11, 24) }
.fsx (fsharp)
__assert_eq { expected = (1, 2, 6, 11, 24) }
.ts (typescript)
__assert_eq { expected = (1, 2, 6, 11, 24) }
.py (python)
__assert_eq { expected = (1, 2, 6, 11, 24) }
.lua (lua)
__assert_eq { expected = (1, 2, 6, 11, 24) }
.c (c)
__assert_eq { expected = (1, 2, 6, 11, 24) }
.zig (zig)
__assert_eq { expected = (1, 2, 6, 11, 24) }
.gleam (gleam)
__assert_eq { expected = (1, 2, 6, 11, 24) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### a_raw_number_is_not_a_bound (test)

In [6]:
///- --test --fails ETY-TermError "Unification failure"
///> zig

dice_count 36u64

failed to compile as expected (ETY-TermError "Unification failure"): FatalError: File main has a type error somewhere in its path. /home/runner/work/dice/dice/lib/dice.spi:177:16 (main.spi:86:16): ETY-TermError: Unification failure. Got:      bound _ Expected: u64 In the definition: spiral_cells_test.


## hand

In [ ]:
:= hand_value \* t {number}. (b : bound t) (count : i32) (hand : list face) : option t =
    := max = bound_value b
    :~ ^ go (left : i32) (faces : list face) (value : t) : option t =
        ? left <= 0
        ?> ? value < max ?> Some (value + 1) |: None
        |:
            ?= faces ?|
            | f :: rest =>
                := d = digit f
                ? d >= max || value > (max - 1 - d) / 6 ?> None |: go (left - 1) rest (value * 6 + d)
            | [] => None
    go count hand 0

:= faces_of_pips (rolls : list u8) : option (list face) =
    :~ ^ go (rolls : list u8) (acc : list face) : option (list face) =
        ?= rolls ?|
        | r :: rest =>
            ?= face_of_pips r ?|
            | Some f => go rest (f :: acc)
            | None => None
        | [] => Some (listm.rev acc)
    go rolls []

:= roll_within_bounds \* t {number}. (max : t) (rolls : list u8) : option t =
    ?= bound_of max, faces_of_pips rolls ?|
    | Ok b, Some hand => hand_value b (listm.length hand) hand
    | _ => None

#### roll_within_bounds (test)

In [7]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

roll_within_bounds 2000i32 [ 1; 5; 4; 4; 5 ]
|> _assert_eq (Some 995)

.rs (rust)
__assert_eq { expected = Some(995) }
.fsx (fsharp)
__assert_eq { expected = Some(995) }
.ts (typescript)
__assert_eq { expected = Some(995) }
.py (python)
__assert_eq { expected = Some(995) }
.lua (lua)
__assert_eq { expected = Some(995) }
.c (c)
__assert_eq { expected = Some(995) }
.zig (zig)
__assert_eq { expected = Some(995) }
.gleam (gleam)
__assert_eq { expected = Some(995) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

In [8]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

roll_within_bounds 2000i32 [ 2; 2; 6; 4; 5 ]
|> _assert_eq (Some 1715)

.rs (rust)
__assert_eq { expected = Some(1715) }
.fsx (fsharp)
__assert_eq { expected = Some(1715) }
.ts (typescript)
__assert_eq { expected = Some(1715) }
.py (python)
__assert_eq { expected = Some(1715) }
.lua (lua)
__assert_eq { expected = Some(1715) }
.c (c)
__assert_eq { expected = Some(1715) }
.zig (zig)
__assert_eq { expected = Some(1715) }
.gleam (gleam)
__assert_eq { expected = Some(1715) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

In [9]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

roll_within_bounds 2000i32 [ 4; 1; 1; 2; 3 ]
|> _assert_eq None

.rs (rust)
__assert_eq { expected = None }
.fsx (fsharp)
__assert_eq { expected = None }
.ts (typescript)
__assert_eq { expected = None }
.py (python)
__assert_eq { expected = None }
.lua (lua)
__assert_eq { expected = None }
.c (c)
__assert_eq { expected = None }
.zig (zig)
__assert_eq { expected = None }
.gleam (gleam)
__assert_eq { expected = None }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### roll_within_bounds_rejects_a_face_that_no_die_has (test)

In [10]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

(roll_within_bounds 2000i32 [ 1; 7; 1 ], roll_within_bounds 2000i32 [ 0; 1 ], roll_within_bounds 0i32 [ 1 ])
|> _assert_eq (None, None, None)

.rs (rust)
__assert_eq { expected = (None, None, None) }
.fsx (fsharp)
__assert_eq { expected = (None, None, None) }
.ts (typescript)
__assert_eq { expected = (None, None, None) }
.py (python)
__assert_eq { expected = (None, None, None) }
.lua (lua)
__assert_eq { expected = (None, None, None) }
.c (c)
__assert_eq { expected = (None, None, None) }
.zig (zig)
__assert_eq { expected = (None, None, None) }
.gleam (gleam)
__assert_eq { expected = (None, None, None) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### roll_within_bounds_never_overflows_a_long_hand (test)

In [11]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

roll_within_bounds 4738381338321616896u64 [ 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6; 6 ]
|> _assert_eq None

.rs (rust)
__assert_eq { expected = None }
.fsx (fsharp)
__assert_eq { expected = None }
.ts (typescript)
__assert_eq { expected = None }
.py (python)
__assert_eq { expected = None }
.lua (lua)
__assert_eq { expected = None }
.c (c)
__assert_eq { expected = None }
.zig (zig)
__assert_eq { expected = None }
.gleam (gleam)
__assert_eq { expected = None }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

In [ ]:
:~ ^ faces (from : u8) : list u8 = ? from > 6 ?> [] |: from :: faces (from + 1)

:~ ^ pair_results (firsts : list u8) (acc : list (option i32)) : list (option i32) =
    ?= firsts ?|
    | a :: rest => pair_results rest (listm.fold (\ acc b => roll_within_bounds 20i32 [ a; b ] :: acc) acc (faces 1))
    | [] => acc

:~ ^ occurrences (v : i32) (results : list (option i32)) (n : i32) : i32 =
    ?= results ?|
    | Some x :: rest => occurrences v rest (? x = v ?> n + 1 |: n)
    | None :: rest => occurrences v rest n
    | [] => n

:~ ^ values_seen_once (v : i32) (results : list (option i32)) : bool =
    ? v > 20i32 ?> !+ |: occurrences v results 0 = 1 && values_seen_once (v + 1) results

#### every_two_roll_sequence_maps_to_one_value_so_accepted_rolls_are_uniform (test)

In [12]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

:= results = pair_results (faces 1) []
:= rejected = listm.fold (\ n r => ?= r ?| Some _ => n | None => n + 1i32) 0i32 results
(values_seen_once 1i32 results, rejected) |> _assert_eq (!+, 16i32)

.rs (rust)
__assert_eq { expected = (true, 16) }
.fsx (fsharp)
__assert_eq { expected = (true, 16) }
.ts (typescript)
__assert_eq { expected = (true, 16) }
.py (python)
__assert_eq { expected = (true, 16) }
.lua (lua)
__assert_eq { expected = (true, 16) }
.c (c)
__assert_eq { expected = (true, 16) }
.zig (zig)
__assert_eq { expected = (true, 16) }
.gleam (gleam)
__assert_eq { expected = (true, 16) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

## sources

In [ ]:
:? draw_face s : s -> face * s

:! replay = { all : list face; rest : list face }

:= replayed (faces : list face) : replay =
    replay { all = faces; rest = faces }

:+ draw_face replay =
    \ (replay { all rest }) =>
        ?= rest ?|
        | f :: rest => f, replay { all = all; rest = rest }
        | [] =>
            ?= all ?|
            | f :: rest => f, replay { all = all; rest = rest }
            | [] => One, replay { all = all; rest = [] }

:! system = ()

:+ draw_face system =
    \ s => face_of_digit (random' 0i32 6i32), s

:! roller = () -> u8

:~ ^ draw_rolled (roll : () -> u8) : face =
    ?= face_of_pips (roll ()) ?|
    | Some f => f
    | None => draw_rolled roll

:+ draw_face roller =
    \ (roller roll) => draw_rolled roll, roller roll

:! hash_faces h = { hasher : h; digest : array u8; at : i32 }

:= hashed \* h. (hasher : h) (digest : array u8) : hash_faces h =
    hash_faces { hasher = hasher; digest = digest; at = 0 }

:~ ^ draw_hashed \* h {digest_with}. (hasher : h) (digest : array u8) (at : i32) : face * hash_faces h =
    ? at >= length digest
    ?> draw_hashed hasher (digest_with hasher digest) 0
    |:
        ?= face_of_byte (index digest at) ?|
        | Some f => f, hash_faces { hasher = hasher; digest = digest; at = at + 1 }
        | None => draw_hashed hasher digest (at + 1)

:+ draw_face hash_faces h {digest_with} =
    \ (hash_faces { hasher digest at }) => draw_hashed hasher digest at

## roll

In [ ]:
:| reroll =
    | Fresh
    | Slide
    | Recycle

:~ ^ roll_hand \* s {draw_face}. (count : i32) (hand : list face) (source : s) : list face * s =
    ? count <= 0
    ?> hand, source
    |:
        := f, source = draw_face source
        roll_hand (count - 1) (f :: hand) source

:~ ^ roll_slid \* t {number} s {draw_face}. (b : bound t) (count : i32) (hand : list face) (source : s) : t * s =
    ?= hand_value b count hand ?|
    | Some value => value, source
    | None =>
        := f, source = draw_face source
        roll_slid b count (f :: hand) source

:~ ^ roll_fresh \* t {number} s {draw_face}. (b : bound t) (count : i32) (source : s) : t * s =
    :~ ^ read (left : i32) (place : t) (value : t) (source : s) : t * s =
        ? left <= 0
        ?> value, source
        |:
            := f, source = draw_face source
            read (left - 1) (place * 6) (value + digit f * place) source
    := value, source = read count 1 0 source
    ? value < bound_value b ?> value + 1, source |: roll_fresh b count source

:! pool t = { value : t; range : t }

:= empty_pool \* t {number}. () : pool t =
    pool { value = 0; range = 1 }

:= settle \* t {number}. (b : bound t) ((pool { value range }) : pool t) : result t (pool t) =
    := max = bound_value b
    := whole = range / max * max
    ? value < whole ?> Ok (value % max + 1) |: Error (pool { value = value - whole; range = range - whole })

:= with_die \* t {number}. ((pool { value range }) : pool t) (f : face) : pool t =
    ? range > (limit.max : t) / 6
    ?> pool { value = digit f; range = 6 }
    |: pool { value = value * 6 + digit f; range = range * 6 }

:~ ^ roll_recycled \* t {number} s {draw_face}. (b : bound t) (p : pool t) (source : s) : t * s =
    ?= settle b p ?|
    | Ok value => value, source
    | Error p =>
        := f, source = draw_face source
        roll_recycled b (with_die p f) source

:= roll \* t {number} s {draw_face}. (policy : reroll) (b : bound t) (source : s) : t * s =
    := count = dice_count b
    ?= policy ?|
    | Recycle => roll_recycled b (empty_pool ()) source
    | Fresh => roll_fresh b count source
    | Slide =>
        := hand, source = roll_hand count [] source
        roll_slid b count hand source

:= roll_value \* t {number} s {draw_face}. (policy : reroll) (max : t) (source : s) : t =
    ?= bound_of max ?|
    | Ok b => fst (roll policy b source)
    | Error e => failwith (bound_error_text e)

#### a_replayed_hand_reads_its_newest_die_first (test)

In [13]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

roll_value Slide 2000i32 (replayed [ Five; Four; Four; Five; One ])
|> _assert_eq 995

.rs (rust)
__assert_eq { expected = 995 }
.fsx (fsharp)
__assert_eq { expected = 995 }
.ts (typescript)
__assert_eq { expected = 995 }
.py (python)
__assert_eq { expected = 995 }
.lua (lua)
__assert_eq { expected = 995 }
.c (c)
__assert_eq { expected = 995 }
.zig (zig)
__assert_eq { expected = 995 }
.gleam (gleam)
__assert_eq { expected = 995 }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### sliding_keeps_the_newest_dice_after_a_rejection (test)

In [14]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

roll_value Slide 2000i32 (replayed [ Five; Four; Four; Five; Two; Two ])
|> _assert_eq 1678

.rs (rust)
__assert_eq { expected = 1678 }
.fsx (fsharp)
__assert_eq { expected = 1678 }
.ts (typescript)
__assert_eq { expected = 1678 }
.py (python)
__assert_eq { expected = 1678 }
.lua (lua)
__assert_eq { expected = 1678 }
.c (c)
__assert_eq { expected = 1678 }
.zig (zig)
__assert_eq { expected = 1678 }
.gleam (gleam)
__assert_eq { expected = 1678 }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### the_largest_bound_accepts_the_highest_hand (test)

In [15]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

roll_value Fresh 4738381338321616896u64 (replayed [ Six ])
|> _assert_eq 4738381338321616896u64

.rs (rust)
__assert_eq { expected = 4738381338321616896 }
.fsx (fsharp)
__assert_eq { expected = 4738381338321616896 }
.ts (typescript)
__assert_eq { expected = 4738381338321616896 }
.py (python)
__assert_eq { expected = 4738381338321616896 }
.lua (lua)
__assert_eq { expected = 4738381338321616896 }
.c (c)
__assert_eq { expected = 4738381338321616896 }
.zig (zig)
__assert_eq { expected = 4738381338321616896 }
.gleam (gleam)
__assert_eq { expected = 4738381338321616896 }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### a_single_die_bound_keeps_only_the_lowest_face (test)

In [16]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

roll_value Fresh 1i32 (replayed [ Six; Four; One ])
|> _assert_eq 1i32

.rs (rust)
__assert_eq { expected = 1 }
.fsx (fsharp)
__assert_eq { expected = 1 }
.ts (typescript)
__assert_eq { expected = 1 }
.py (python)
__assert_eq { expected = 1 }
.lua (lua)
__assert_eq { expected = 1 }
.c (c)
__assert_eq { expected = 1 }
.zig (zig)
__assert_eq { expected = 1 }
.gleam (gleam)
__assert_eq { expected = 1 }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### fresh_rerolls_every_die_after_a_rejection (test)

In [17]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

roll_value Fresh 20i32 (replayed [ Six; Six; Two; One ])
|> _assert_eq 2i32

.rs (rust)
__assert_eq { expected = 2 }
.fsx (fsharp)
__assert_eq { expected = 2 }
.ts (typescript)
__assert_eq { expected = 2 }
.py (python)
__assert_eq { expected = 2 }
.lua (lua)
__assert_eq { expected = 2 }
.c (c)
__assert_eq { expected = 2 }
.zig (zig)
__assert_eq { expected = 2 }
.gleam (gleam)
__assert_eq { expected = 2 }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### a_source_without_draw_face_cannot_roll (test)

In [18]:
///- --test --fails ETY-InstanceNotFound "draw_face"
///> zig

roll_value Fresh 6i32 (bound 6i32)

failed to compile as expected (ETY-InstanceNotFound "draw_face"): FatalError: File main has a type error somewhere in its path. /home/runner/work/dice/dice/lib/dice.spi:484:28 (main.spi:238:28): ETY-InstanceNotFound: The higher order type instance bound does not have the prototype draw_face. In the definition: spiral_cells_test.


In [ ]:
:~ ^ hashed_values (left : i32) (source : hash_faces keccak.spiral_keccak) (acc : list u64) : list u64 =
    ? left <= 0
    ?> listm.rev acc
    |:
        ?= bound_of 2000u64 ?|
        | Ok b =>
            := value, source = roll Fresh b source
            hashed_values (left - 1) source (value :: acc)
        | Error _ => acc

#### a_keccak_stream_rolls_the_same_values_on_every_backend (test)

In [19]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

:= ~seed_size = 3i32
:= seed : array u8 = bytes.counting seed_size
hashed_values 6 (hashed keccak.spiral_keccak seed) []
|> _assert_eq [ 1295u64; 295; 1135; 1360; 1663; 1972 ]

.rs (rust)
__assert_eq { expected = Cons(1295, Cons(295, Cons(1135, Cons(1360, Cons(1663, Cons(1972, Nil)))))) }
.fsx (fsharp)
__assert_eq { expected = Cons(1295, Cons(295, Cons(1135, Cons(1360, Cons(1663, Cons(1972, Nil)))))) }
.ts (typescript)
__assert_eq { expected = Cons(1295, Cons(295, Cons(1135, Cons(1360, Cons(1663, Cons(1972, Nil)))))) }
.py (python)
__assert_eq { expected = Cons(1295, Cons(295, Cons(1135, Cons(1360, Cons(1663, Cons(1972, Nil)))))) }
.lua (lua)
__assert_eq { expected = Cons(1295, Cons(295, Cons(1135, Cons(1360, Cons(1663, Cons(1972, Nil)))))) }
.c (c)
__assert_eq { expected = Cons(1295, Cons(295, Cons(1135, Cons(1360, Cons(1663, Cons(1972, Nil)))))) }
.zig (zig)
__assert_eq { expected = Cons(1295, Cons(295, Cons(1135, Cons(1360, Cons(1663, Cons(1972, Nil)))))) }
.gleam (gleam)
__assert_eq { expected = Cons(1295, Cons(295, Cons(1135, Cons(1360, Cons(1663, Cons(1972, Nil)))))) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

In [ ]:
:~ ^ histogram_misses (policy : reroll) (max : i32) (draws : i32) (seen : list i32) : i32 =
    ? draws <= 0
    ?> listm'.init_series 1i32 max 1i32 |> listm.fold (\ misses v => ? listm.fold (\ found x => found || x = v) !- seen ?> misses |: misses + 1) 0i32
    |:
        := value = roll_value policy max system
        ? value < 1 || value > max ?> -1i32 |: histogram_misses policy max (draws - 1) (value :: seen)

#### system_rolls_cover_every_value_and_stay_in_range (test)

In [20]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua

:= ~draws = 3000i32
(histogram_misses Fresh 10i32 draws [], histogram_misses Slide 10i32 draws [], histogram_misses Fresh 37i32 draws [])
|> _assert_eq (0i32, 0i32, 0i32)

.rs (rust)
__assert_eq { expected = (0, 0, 0) }
.fsx (fsharp)
__assert_eq { expected = (0, 0, 0) }
.ts (typescript)
__assert_eq { expected = (0, 0, 0) }
.py (python)
__assert_eq { expected = (0, 0, 0) }
.lua (lua)
__assert_eq { expected = (0, 0, 0) }

0

In [ ]:
:! counted s = { inner : s; drawn : i32 }

:+ draw_face counted s {draw_face} =
    \ (counted { inner drawn }) =>
        := f, inner = draw_face inner
        f, counted { inner = inner; drawn = drawn + 1 }

:~ ^ recycled_outcome \* t {number}. (b : bound t) (p : pool t) (faces : list face) : option t =
    ?= settle b p ?|
    | Ok value => Some value
    | Error p =>
        ?= faces ?|
        | f :: rest => recycled_outcome b (with_die p f) rest
        | [] => None

:= outcome_spread (max : i32) : i32 * i32 * i32 =
    := counts : array i32 = create (max + 1)
    :~ ^ clear (i : i32) : () =
        ? i <= max ?>
            set counts i 0
            clear (i + 1)
    clear 0
    ?= bound_of max ?|
    | Ok b =>
        :~ ^ tally (sequence : i32) : () =
            ? sequence < 216 ?>
                := faces = [ face_of_digit (sequence / 36); face_of_digit (sequence / 6 % 6); face_of_digit (sequence % 6) ]
                := slot =
                    ?= recycled_outcome b (empty_pool ()) faces ?|
                    | Some value => value
                    | None => 0
                set counts slot (index counts slot + 1)
                tally (sequence + 1)
        tally 0
    | Error _ => ()
    :~ ^ spread (i : i32) (low : i32) (high : i32) : i32 * i32 =
        ? i > max ?> low, high |: spread (i + 1) (? index counts i < low ?> index counts i |: low) (? index counts i > high ?> index counts i |: high)
    := low, high = spread 1 1000 0
    low, high, index counts 0

#### recycling_is_exact_over_every_three_dice_sequence (test)

In [21]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

:= ~five = 5i32
(outcome_spread five, outcome_spread (five + 2), outcome_spread (five * 4))
|> _assert_eq ((43i32, 43i32, 1i32), (30i32, 30i32, 6i32), (10i32, 10i32, 16i32))

.rs (rust)
__assert_eq { expected = ((43, 43, 1), (30, 30, 6), 10, 10, 16) }
.fsx (fsharp)
__assert_eq { expected = ((43, 43, 1), (30, 30, 6), 10, 10, 16) }
.ts (typescript)
__assert_eq { expected = ((43, 43, 1), (30, 30, 6), 10, 10, 16) }
.py (python)
__assert_eq { expected = ((43, 43, 1), (30, 30, 6), 10, 10, 16) }
.lua (lua)
__assert_eq { expected = ((43, 43, 1), (30, 30, 6), 10, 10, 16) }
.c (c)
__assert_eq { expected = ((43, 43, 1), (30, 30, 6), 10, 10, 16) }
.zig (zig)
__assert_eq { expected = ((43, 43, 1), (30, 30, 6), 10, 10, 16) }
.gleam (gleam)
__assert_eq { expected = ((43, 43, 1), (30, 30, 6), 10, 10, 16) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

In [ ]:
:~ ^ recycled_values (left : i32) (source : hash_faces keccak.spiral_keccak) (acc : list u64) : list u64 =
    ? left <= 0
    ?> listm.rev acc
    |:
        ?= bound_of 2000u64 ?|
        | Ok b =>
            := value, source = roll Recycle b source
            recycled_values (left - 1) source (value :: acc)
        | Error _ => acc

#### a_recycled_keccak_stream_rolls_the_same_values_on_every_backend (test)

In [22]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

:= ~seed_size = 3i32
recycled_values 6 (hashed keccak.spiral_keccak (bytes.counting seed_size)) []
|> _assert_eq [ 321u64; 1537; 145; 1654; 1384; 761 ]

.rs (rust)
__assert_eq { expected = Cons(321, Cons(1537, Cons(145, Cons(1654, Cons(1384, Cons(761, Nil)))))) }
.fsx (fsharp)
__assert_eq { expected = Cons(321, Cons(1537, Cons(145, Cons(1654, Cons(1384, Cons(761, Nil)))))) }
.ts (typescript)
__assert_eq { expected = Cons(321, Cons(1537, Cons(145, Cons(1654, Cons(1384, Cons(761, Nil)))))) }
.py (python)
__assert_eq { expected = Cons(321, Cons(1537, Cons(145, Cons(1654, Cons(1384, Cons(761, Nil)))))) }
.lua (lua)
__assert_eq { expected = Cons(321, Cons(1537, Cons(145, Cons(1654, Cons(1384, Cons(761, Nil)))))) }
.c (c)
__assert_eq { expected = Cons(321, Cons(1537, Cons(145, Cons(1654, Cons(1384, Cons(761, Nil)))))) }
.zig (zig)
__assert_eq { expected = Cons(321, Cons(1537, Cons(145, Cons(1654, Cons(1384, Cons(761, Nil)))))) }
.gleam (gleam)
__assert_eq { expected = Cons(321, Cons(1537, Cons(145, Cons(1654, Cons(1384, Cons(761, Nil)))))) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

In [ ]:
:~ ^ faces_drawn (policy : reroll) (b : bound u64) (left : i32) (source : counted (hash_faces keccak.spiral_keccak)) : i32 =
    ? left <= 0
    ?> (\ (counted { inner drawn }) => drawn) source
    |:
        := _, source = roll policy b source
        faces_drawn policy b (left - 1) source

:= faces_for (policy : reroll) (max : u64) : i32 =
    := source = counted { inner = hashed keccak.spiral_keccak (bytes.counting 3i32); drawn = 0 }
    ?= bound_of max ?|
    | Ok b => faces_drawn policy b 20 source
    | Error _ => -1

#### recycling_draws_fewer_dice_than_rerolling (test)

In [23]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

:= ~max = 2000u64
(faces_for Fresh max, faces_for Recycle max, faces_for Fresh 46657u64, faces_for Recycle 46657u64, faces_for Fresh 217u64, faces_for Recycle 217u64)
|> _assert_eq (320i32, 106i32, 994i32, 141i32, 568i32, 88i32)

.rs (rust)
__assert_eq { expected = (320, 106, 994, 141, 568, 88) }
.fsx (fsharp)
__assert_eq { expected = (320, 106, 994, 141, 568, 88) }
.ts (typescript)
__assert_eq { expected = (320, 106, 994, 141, 568, 88) }
.py (python)
__assert_eq { expected = (320, 106, 994, 141, 568, 88) }
.lua (lua)
__assert_eq { expected = (320, 106, 994, 141, 568, 88) }
.c (c)
__assert_eq { expected = (320, 106, 994, 141, 568, 88) }
.zig (zig)
__assert_eq { expected = (320, 106, 994, 141, 568, 88) }
.gleam (gleam)
__assert_eq { expected = (320, 106, 994, 141, 568, 88) }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

#### recycling_reaches_the_largest_bound_without_overflow (test)

In [24]:
///- --test
///> rust
///> fsharp
///> typescript
///> python
///> lua
///> c
///> zig
///> gleam
///> pascal

roll_value Recycle 4738381338321616896u64 (replayed [ Six ])
|> _assert_eq 4738381338321616896u64

.rs (rust)
__assert_eq { expected = 4738381338321616896 }
.fsx (fsharp)
__assert_eq { expected = 4738381338321616896 }
.ts (typescript)
__assert_eq { expected = 4738381338321616896 }
.py (python)
__assert_eq { expected = 4738381338321616896 }
.lua (lua)
__assert_eq { expected = 4738381338321616896 }
.c (c)
__assert_eq { expected = 4738381338321616896 }
.zig (zig)
__assert_eq { expected = 4738381338321616896 }
.gleam (gleam)
__assert_eq { expected = 4738381338321616896 }
pascal: unavailable on this machine (`dcc32` was not found (nor fpc))

0

## rotate_number

In [ ]:
:= rotate_number \* t {number}. (max : i64) (n : t) : t =
    (conv n - 1 + max) % max + 1 |> conv

:= rotate_numbers max items =
    items |> stream.map (rotate_number max)

#### rotate_numbers (test)

In [25]:
///- --test
///> rust
///> fsharp
///> typescript

listm'.init_series -1 14 1i32
|> stream.from_list
|> rotate_numbers 6
|> stream.to_list
|> _assert_eq [ 5; 6; 1; 2; 3; 4; 5; 6; 1; 2; 3; 4; 5; 6; 1; 2 ]

.rs (rust)
__assert_eq { expected = Cons(5, Cons(6, Cons(1, Cons(2, Cons(3, Cons(4, Cons(5, Cons(6, Cons(1, Cons(2, Cons(3, Cons(4, Cons(5, Cons(6, Cons(1, Cons(2, Nil)))))))))))))))) }
.fsx (fsharp)
__assert_eq { expected = Cons(5, Cons(6, Cons(1, Cons(2, Cons(3, Cons(4, Cons(5, Cons(6, Cons(1, Cons(2, Cons(3, Cons(4, Cons(5, Cons(6, Cons(1, Cons(2, Nil)))))))))))))))) }
.ts (typescript)
__assert_eq { expected = Cons(5, Cons(6, Cons(1, Cons(2, Cons(3, Cons(4, Cons(5, Cons(6, Cons(1, Cons(2, Cons(3, Cons(4, Cons(5, Cons(6, Cons(1, Cons(2, Nil)))))))))))))))) }

0

## create_sequential_roller

In [ ]:
:= create_sequential_roller list =
    := s = list |> listm.rev |> listm.append list |> stream.from_list |> stream.memoize
    := current_index = mut 0i64
    := acc = mut 1
    := len = mut -1
    :~ ^ 루프 () =
        ?= s () |> stream.try_item *current_index ?|
        | Some item =>
            current_index <- *current_index + 1
            item
        | None =>
            ? *len = -1
            ?> len <- *current_index
            acc <-
                ? *acc >= *len
                ?> 1
                |: *acc + 1
            current_index <- *acc - 1
            루프 ()
    루프

In [26]:
///- --test
///> rust
///> fsharp
///> typescript

:= sequential_roll = create_sequential_roller [ 1i32; 2; 3; 4 ]

arraym.init 50i32 (ignore >> sequential_roll)
|> _assert_eq (;[ 1; 2; 3; 4; 4; 3; 2; 1; 2; 3; 4; 4; 3; 2; 1; 3; 4; 4; 3; 2; 1; 4; 4; 3; 2; 1;
4; 3; 2; 1; 3; 2; 1; 2; 1; 1; 1; 2; 3; 4; 4; 3; 2; 1; 2; 3; 4; 4; 3; 2 ] : array i32)

.rs (rust)
__assert_eq { expected = [1, 2, 3, 4, 4, 3, 2, 1, 2, 3, 4, 4, 3, 2, 1, 3, 4, 4, 3, 2, 1, 4, 4, 3, 2, 1, 4, 3, 2, 1, 3, 2, 1, 2, 1, 1, 1, 2, 3, 4, 4, 3, 2, 1, 2, 3, 4, 4, 3, 2] }
.fsx (fsharp)
__assert_eq { expected = [1, 2, 3, 4, 4, 3, 2, 1, 2, 3, 4, 4, 3, 2, 1, 3, 4, 4, 3, 2, 1, 4, 4, 3, 2, 1, 4, 3, 2, 1, 3, 2, 1, 2, 1, 1, 1, 2, 3, 4, 4, 3, 2, 1, 2, 3, 4, 4, 3, 2] }
.ts (typescript)
__assert_eq { expected = [1, 2, 3, 4, 4, 3, 2, 1, 2, 3, 4, 4, 3, 2, 1, 3, 4, 4, 3, 2, 1, 4, 4, 3, 2, 1, 4, 3, 2, 1, 3, 2, 1, 2, 1, 1, 1, 2, 3, 4, 4, 3, 2, 1, 2, 3, 4, 4, 3, 2] }

0

## roll_progressively

In [ ]:
:= roll_progressively (roll : () -> u8) (reroll : bool) max =
    roll_value (? reroll ?> Fresh |: Slide) max (roller roll)

In [27]:
///- --test
///> rust
///> fsharp
///> typescript

:= sequential_roll = create_sequential_roller [ 5u8; 4; 4; 5; 1 ]
roll_progressively sequential_roll !- 2000i32
|> _assert_eq 995

.rs (rust)
__assert_eq { expected = 995 }
.fsx (fsharp)
__assert_eq { expected = 995 }
.ts (typescript)
__assert_eq { expected = 995 }

0

In [28]:
///- --test
///> rust
///> fsharp
///> typescript

:= sequential_roll = create_sequential_roller [ 5u8; 4; 4; 5; 2 ]
roll_progressively sequential_roll !- 2000i32
|> _assert_eq 1678

.rs (rust)
__assert_eq { expected = 1678 }
.fsx (fsharp)
__assert_eq { expected = 1678 }
.ts (typescript)
__assert_eq { expected = 1678 }

0

## main

In [ ]:
:= main (_args : array_base string) =
    := result = roll_value Fresh 4738381338321616896i64 system
    trace Debug
        \ () => "dice.main"
        \ () => { result }
    0i32

:= main () =
    := rotate_numbers' : i64 -> stream.stream u8 -> stream.stream u8 =
        rotate_numbers

    := create_sequential_roller' :
        list u8
        -> (() -> u8)
        =
        create_sequential_roller

    := roll_progressively' :
        (() -> u8)
        -> bool
        -> u64
        -> u64
        =
        roll_progressively

    := roll_within_bounds' :
        u64
        -> list u8
        -> optionm'.option' u64
        =
        \ a b =>
            roll_within_bounds a b
            |> optionm'.box

    backend_switch {
        Fsharp = \ () => $'let rotate_numbers x = !rotate_numbers' x' : ()
        Rust = \ () => ()
        TypeScript = \ () => ()
        C = \ () => ()
    } : ()
    backend_switch {
        Fsharp = \ () => $'let create_sequential_roller x = !create_sequential_roller' x' : ()
        Rust = \ () => ()
        TypeScript = \ () => ()
        C = \ () => ()
    } : ()
    backend_switch {
        Fsharp = \ () => $'let roll_progressively x = !roll_progressively' x' : ()
        Rust = \ () => ()
        TypeScript = \ () => ()
        C = \ () => ()
    } : ()
    backend_switch {
        Fsharp = \ () => $'let roll_within_bounds x = !roll_within_bounds' x' : ()
        Rust = \ () => ()
        TypeScript = \ () => ()
        C = \ () => ()
    } : ()
    backend_switch {
        Fsharp = \ () => $'let main args = !main args' : ()
        Rust = \ () => main (;[] |> am'.base) |> ignore
        Python = \ () => main (;[] |> am'.base) |> ignore
        TypeScript = \ () => main (;[] |> am'.base) |> ignore
        C = \ () => ()
    } : ()